In [3]:
from src.hr_docs import HR_DOCUMENTS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings



EMBED_MODEL   = "BAAI/bge-small-en-v1.5"
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 100

def _make_documents():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    docs = []
    for item in HR_DOCUMENTS:
        chunks = splitter.create_documents(
            texts=[item["content"]],
            metadatas=[{"source": item["title"]}],
        )
        docs.extend(chunks)
    return docs


In [8]:
from src.retriever import NativeFAISSRetriever
import streamlit as st
@st.cache_resource(show_spinner="Building HR policy index… (first run downloads ~40 MB embedding model)")
def build_vectorstore():
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    docs = _make_documents()
    vectorstore = NativeFAISSRetriever.from_documents(documents=docs, embeddings=embeddings, k=4)
    return vectorstore


In [19]:
def retrieve(query: str, vectorstore, k: int = 3):
    """Return top-k chunks with relevance scores."""
    results = vectorstore.invoke(query, k=k)
    print(results)
    return [
        {
            "source": doc.metadata.get("source", "Unknown"),
            "content": doc.page_content,
        }
        for doc in results
    ]


In [20]:
vectorstore = build_vectorstore()
response = retrieve(query="I want to know about annual leave", vectorstore=vectorstore)

[Document(metadata={'source': 'Annual Leave & Time Off Policy', 'similarity_score': 0.7263633012771606}, page_content="Accrual begins on the employee's first day. The leave year runs January 1 to December 31.\nUp to 5 unused vacation days may be carried over to the following year and must be used by March 31.\nUnused carried-over days are forfeited after March 31."), Document(metadata={'source': 'Annual Leave & Time Off Policy', 'similarity_score': 0.7245297431945801}, page_content='Annual Leave & Time Off Policy — Acme Corp'), Document(metadata={'source': 'Annual Leave & Time Off Policy', 'similarity_score': 0.6942645311355591}, page_content="VACATION DAYS\nFull-time employees accrue 15 paid vacation days per year for the first three years of service.\nAfter 3 years of continuous employment, accrual increases to 20 days per year.\nAfter 7 years, accrual increases to 25 days per year.\nPart-time employees accrue vacation on a pro-rata basis relative to their contracted hours.\nAccrual 